# Bayesian inference on the Planck 2018 binned TT power spectrum

**Final project — Advanced data & statistical methods lab.**
Reproducible notebook (Option 1: companion to the LaTeX report).

We fit the publicly released Planck 2018 *binned* temperature (TT) angular
power spectrum with a CAMB-based theoretical model and a simplified per-bin
Gaussian likelihood, sampled with MCMC.

---
## Step 1 — Setup & data loading
Load the real Planck 2018 binned TT bandpowers and plot them.

In [ ]:
# Step 1: imports and configuration
import os
os.environ["OMP_NUM_THREADS"] = "1"   # 1 thread/process; we parallelise over walkers (Step 4)
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "data"
FIG_DIR  = "figures"
CHAIN_DIR = "chains"
for d in (DATA_DIR, FIG_DIR, CHAIN_DIR):
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

### Load the Planck 2018 binned TT spectrum

File `COM_PowerSpect_CMB-TT-binned_R3.01.txt` (Planck PR3 ancillary data).
Five columns: effective multipole $\ell$, bandpower
$D_\ell=\ell(\ell+1)C_\ell/2\pi\;[\mu K^2]$, lower/upper $1\sigma$ errors,
and the Planck best fit. It is downloaded once and cached locally.

In [ ]:
# Step 1: download (once) and load the binned TT spectrum
FNAME = "COM_PowerSpect_CMB-TT-binned_R3.01.txt"
URL = ("https://irsa.ipac.caltech.edu/data/Planck/release_3/"
       "ancillary-data/cosmoparams/" + FNAME)
path = os.path.join(DATA_DIR, FNAME)

if not os.path.exists(path):           # fetch only if not cached
    urllib.request.urlretrieve(URL, path)

# columns: ell, Dl, -dDl, +dDl, BestFit
ell, Dl, dDl_lo, dDl_hi, bestfit = np.loadtxt(path, unpack=True)
err = 0.5 * (dDl_lo + dDl_hi)          # symmetric per-bin 1-sigma error

print(f"{len(ell)} bins, ell range {ell.min():.0f}-{ell.max():.0f}")

In [ ]:
# Step 1: plot the data with error bars
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.errorbar(ell, Dl, yerr=err, fmt="o", ms=3, lw=1,
            color="C0", ecolor="0.6", label="Planck 2018 (binned TT)")
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$D_\ell=\ell(\ell+1)C_\ell/2\pi\ \ [\mu K^2]$")
ax.set_title("Planck 2018 binned TT power spectrum")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "planck_tt_data.pdf"))
plt.show()

---
## Step 2 — Theoretical model with CAMB

The 6 $\Lambda$CDM parameters are
$\theta=(\Omega_b h^2,\ \Omega_c h^2,\ H_0,\ \tau,\ \ln(10^{10}A_s),\ n_s)$.
CAMB returns $D_\ell=\ell(\ell+1)C_\ell/2\pi$ in $\mu K^2$ at integer $\ell$;
we interpolate it to the effective multipoles of the binned data (a simple
approximation to the Planck bin window functions). For speed we fix massless
neutrinos (`mnu=0`) and use low lensing-potential accuracy.

In [ ]:
# Step 2: theoretical binned D_l^TT with CAMB
import camb

LMAX = int(np.ceil(ell.max())) + 50      # compute slightly beyond the data

def theory_dl(theta):
    """Binned theory D_l^TT [muK^2] at the data multipoles.
    theta = [ombh2, omch2, H0, tau, ln10As, ns]."""
    ombh2, omch2, H0, tau, ln10As, ns = theta
    pars = camb.set_params(H0=H0, ombh2=ombh2, omch2=omch2, tau=tau,
                           As=1e-10*np.exp(ln10As), ns=ns,
                           mnu=0.0, lmax=LMAX, lens_potential_accuracy=0)
    res = camb.get_results(pars)
    # raw_cl=False (default): array is already D_l = l(l+1)C_l/2pi
    dl = res.get_cmb_power_spectra(pars, CMB_unit="muK")["total"][:, 0]
    return np.interp(ell, np.arange(dl.size), dl)   # interp to effective l

In [ ]:
# Step 2: sanity check - theory at Planck 2018 fiducial values vs data
# [ombh2, omch2, H0, tau, ln10As, ns]
theta_fid = [0.02237, 0.1200, 67.36, 0.0544, 3.044, 0.9649]
dl_fid = theory_dl(theta_fid)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.errorbar(ell, Dl, yerr=err, fmt="o", ms=3, lw=1, color="C0",
            ecolor="0.6", label="Planck 2018 (binned TT)")
ax.plot(ell, dl_fid, "-", color="C3", lw=1.6, label="CAMB (fiducial)")
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$D_\ell\ [\mu K^2]$")
ax.set_title("Fiducial CAMB model vs Planck data")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "theory_vs_data.pdf"))
plt.show()

---
## Step 3 — Likelihood, priors and posterior

**Likelihood.** We use a simplified Gaussian likelihood that keeps only the
per-bin variance (diagonal covariance), ignoring bin-to-bin correlations:
$$\ln\mathcal{L}(\theta)=-\tfrac12\sum_b
\frac{\bigl(D_b^{\rm data}-D_b^{\rm th}(\theta)\bigr)^2}{\sigma_b^2}.$$

**Priors.** Flat (uniform) priors on five parameters over physically sensible
ranges, and a **Gaussian prior** $\tau=0.0544\pm0.0073$ (Planck lowE) that
breaks the $A_s$–$\tau$ degeneracy unconstrained by TT-only data.

CAMB calls are wrapped in `try/except` so that extreme parameter values that
break the Boltzmann solver simply return $-\infty$ instead of crashing the chain.

In [ ]:
# Step 3: priors, likelihood, posterior
# parameter order: [ombh2, omch2, H0, tau, ln10As, ns]
BOUNDS = np.array([[0.018, 0.026],   # ombh2
                   [0.10,  0.14],    # omch2
                   [55.0,  80.0],    # H0
                   [0.01,  0.12],    # tau
                   [2.7,   3.3],     # ln(1e10 As)
                   [0.92,  1.00]])   # ns
TAU_MU, TAU_SIG = 0.0544, 0.0073     # Planck lowE Gaussian prior on tau
NDIM = BOUNDS.shape[0]

def log_prior(theta):
    if np.any(theta < BOUNDS[:, 0]) or np.any(theta > BOUNDS[:, 1]):
        return -np.inf                       # outside the flat box
    return -0.5 * ((theta[3] - TAU_MU) / TAU_SIG) ** 2   # Gaussian on tau

def log_likelihood(theta):
    try:
        model = theory_dl(theta)             # CAMB may fail on extreme theta
    except Exception:
        return -np.inf
    return -0.5 * np.sum(((Dl - model) / err) ** 2)      # diagonal Gaussian

def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

# quick check: posterior is finite at the fiducial point
print("log-posterior at fiducial:",
      round(log_posterior([0.02237, 0.1200, 67.36, 0.0544, 3.044, 0.9649]), 2))

---
## Step 4 — MCMC sampling with `emcee`

We sample the posterior with the affine-invariant ensemble sampler `emcee`.
Walkers are initialised in a tight Gaussian ball around the Planck fiducial
point (each within the prior box), run through a short burn-in, reset, and
then run for the production phase. The chain is saved to `chains/` so the
analysis steps can reload it without re-sampling.

**Parallelisation.** CAMB dominates the cost, so we parallelise the walker
evaluations with `multiprocessing.Pool` (near-linear speed-up in the number
of cores). Each process runs CAMB single-threaded (`OMP_NUM_THREADS=1`, set
in Step 1) to avoid oversubscription.

> **Runtime / platform note.** Wall time $\approx
> N_\text{walkers}\times(N_\text{burn}+N_\text{prod})\times t_\text{CAMB}/N_\text{cores}$.
> Start small and scale up. The `Pool` works out of the box on Linux; on
> **macOS/Windows** the worker processes may fail to pickle the notebook-defined
> functions — if it hangs or errors, set `USE_POOL = False` (serial) or move
> the model/likelihood functions into a `.py` module and import them.

In [ ]:
# Step 4: sampler configuration and walker initialisation
N_WALKERS = 24       # >= 2*NDIM; more walkers -> better mixing, more evals
N_BURN    = 200      # burn-in steps (discarded)
N_PROD    = 1500     # production steps (kept)

# per-parameter initial spread (small, keeps walkers in-bounds near the peak)
INIT_STD  = np.array([1e-4, 1e-3, 0.5, 0.002, 0.01, 0.005])

rng = np.random.default_rng(42)
p0 = theta_fid + INIT_STD * rng.standard_normal((N_WALKERS, NDIM))
assert np.all([np.isfinite(log_posterior(p)) for p in p0]), "bad init"
print(f"{N_WALKERS} walkers x {N_BURN}+{N_PROD} steps initialised")

In [ ]:
# Step 4: run the sampler (parallel) and save the chain
import emcee
from multiprocessing import Pool, cpu_count

USE_POOL = True      # set False for a serial run (Windows/macOS fallback)

def run_sampler(pool=None):
    s = emcee.EnsembleSampler(N_WALKERS, NDIM, log_posterior, pool=pool)
    state = s.run_mcmc(p0, N_BURN, progress=True)   # burn-in
    s.reset()
    s.run_mcmc(state, N_PROD, progress=True)        # production
    return s

if USE_POOL:
    with Pool() as pool:
        sampler = run_sampler(pool)
else:
    sampler = run_sampler()

chain = sampler.get_chain()                          # (N_PROD, N_WALKERS, NDIM)
log_prob = sampler.get_log_prob()
np.save(os.path.join(CHAIN_DIR, "chain.npy"), chain)
np.save(os.path.join(CHAIN_DIR, "log_prob.npy"), log_prob)
print("mean acceptance fraction:",
      round(float(np.mean(sampler.acceptance_fraction)), 3))

---
## Next steps (to be filled)
- **Step 5** — Convergence diagnostics (autocorrelation, acceptance, traces).
- **Step 6** — Posteriors, credible intervals, best-fit overlay.
- **Step 7** — Model variations / robustness.